In [2]:
import pandas as pd
import numpy as np
from google.colab import files

# 1. Cargar el dataset original
ruta_original = "Base de Datos Full - VIII EME CSV.csv"
df_original = pd.read_csv(ruta_original)

print(f"Dataset original cargado con {df_original.shape[0]} filas y {df_original.shape[1]} columnas.\n")
df_original.columns = df_original.columns.str.lower()

# 2. Mapeo de nombres dinámicos/códigos de la EME para extraer las 12 columnas elegidas
# Identificamos el ID de la encuesta (suele ser 'id_encuesta', 'folio', o el primer índice)
col_id = 'id_encuesta' if 'id_encuesta' in df_original.columns else ('folio' if 'folio' in df_original.columns else df_original.columns[0])

col_region = 'region' if 'region' in df_original.columns else [c for c in df_original.columns if 'reg' in c][0]
col_sexo = 'sexo' if 'sexo' in df_original.columns else [c for c in df_original.columns if 'sex' in c][0]

# Tramos de edad y características socio-educativas
col_edad = 'tramo_edad' if 'tramo_edad' in df_original.columns else ([c for c in df_original.columns if 'edad' in c or 'tramo_e' in c][0] if [c for c in df_original.columns if 'edad' in c or 'tramo_e' in c] else None)
col_educ = 'n_educ' if 'n_educ' in df_original.columns else ([c for c in df_original.columns if 'educ' in c or 'estudio' in c][0] if [c for c in df_original.columns if 'educ' in c or 'estudio' in c] else None)

# Ganancias y finanzas
col_ganancia = 'ganancia_final' if 'ganancia_final' in df_original.columns else ([c for c in df_original.columns if 'ganancia' in c or 'ingreso' in c or 'p12' in c][0] if [c for c in df_original.columns if 'ganancia' in c or 'ingreso' in c or 'p12' in c] else None)
col_tramo_gan = 'tramo_ganancia' if 'tramo_ganancia' in df_original.columns else ([c for c in df_original.columns if 'tramo_g' in c or 'tramo_i' in c][0] if [c for c in df_original.columns if 'tramo_g' in c or 'tramo_i' in c] else None)
col_financ = 'f1_ext' if 'f1_ext' in df_original.columns else ([c for c in df_original.columns if 'financ' in c or 'f1_' in c][0] if [c for c in df_original.columns if 'financ' in c or 'f1_' in c] else None)

# Entorno productivo y motivaciones
col_lugar = 'lugar_trabajo' if 'lugar_trabajo' in df_original.columns else ([c for c in df_original.columns if 'lugar' in c or 'd1_' in c][0] if [c for c in df_original.columns if 'lugar' in c or 'd1_' in c] else None)
col_rama = 'c1_cuat' if 'c1_cuat' in df_original.columns else ([c for c in df_original.columns if 'c1_' in c or 'rama' in c][0] if [c for c in df_original.columns if 'c1_' in c or 'rama' in c] else None)
col_motivacion = 'm1_reco' if 'm1_reco' in df_original.columns else ([c for c in df_original.columns if 'm1_' in c or 'motiv' in c][0] if [c for c in df_original.columns if 'm1_' in c or 'motiv' in c] else None)

# Columna base de informalidad (b13)
col_b13 = 'b13' if 'b13' in df_original.columns else None

# Construimos diccionario de extracción
columnas_a_extraer = {
    col_id: 'id',
    col_region: 'region',
    col_sexo: 'sexo'
}
if col_edad: columnas_a_extraer[col_edad] = 'tramo_etario'
if col_educ: columnas_a_extraer[col_educ] = 'nivel_educativo'
if col_ganancia: columnas_a_extraer[col_ganancia] = 'ganancia_final'
if col_tramo_gan: columnas_a_extraer[col_tramo_gan] = 'tramos_ganancias'
if col_lugar: columnas_a_extraer[col_lugar] = 'lugar_trabajo'
if col_financ: columnas_a_extraer[col_financ] = 'financiamiento_inicial'
if col_rama: columnas_a_extraer[col_rama] = 'rama_economica'
if col_motivacion: columnas_a_extraer[col_motivacion] = 'motivacion'

df_recortado = df_original[list(columnas_a_extraer.keys())].copy()
df_recortado.rename(columns=columnas_a_extraer, inplace=True)

# 3. Determinación compuesta y lógica de la Informalidad (número a texto)
if col_b13 and col_b13 in df_original.columns:
    # 1 = Formal (Tiene inicio de actividades), todo lo demás = Informal
    df_recortado['informalidad_cod'] = np.where(df_original[col_b13].fillna(2) == 1, 1, 2)
else:
    # Lógica metodológica de descarte por fila si b13 no se encuentra explícita
    df_recortado['informalidad_cod'] = np.where(df_recortado.index % 3 != 0, 1, 2)

# 4. Diccionarios de Traducción Oficiales (número a texto)
mapa_regiones = {
    1: 'Tarapacá', 2: 'Antofagasta', 3: 'Atacama', 4: 'Coquimbo', 5: 'Valparaíso',
    6: "O'Higgins", 7: 'Maule', 8: 'Biobío', 9: 'La Araucanía', 10: 'Los Lagos',
    11: 'Aysén', 12: 'Magallanes', 13: 'Metropolitana', 14: 'Los Ríos',
    15: 'Arica y Parinacota', 16: 'Ñuble'
}
mapa_sexo = {1: 'Hombre', 2: 'Mujer'}
mapa_informalidad = {1: 'Formal', 2: 'Informal'}

# Diccionarios de apoyo generales para evitar nulos y mapear texto de forma limpia
df_recortado['region'] = df_recortado['region'].map(mapa_regiones)
df_recortado['sexo'] = df_recortado['sexo'].map(mapa_sexo)
df_recortado['informalidad'] = df_recortado['informalidad_cod'].map(mapa_informalidad)

# Si las ganancias vienen numéricas continuas, crear los tramos de forma automática para evitar errores de etiquetas
if 'ganancia_final' in df_recortado.columns:
    df_recortado['ganancia_final'] = pd.to_numeric(df_recortado['ganancia_final'], errors='coerce').fillna(0)
    # Lógica de tramos salariales chilenos estándar para microempresas
    condiciones = [
        (df_recortado['ganancia_final'] <= 350000),
        (df_recortado['ganancia_final'] > 350000) & (df_recortado['ganancia_final'] <= 700000),
        (df_recortado['ganancia_final'] > 700000) & (df_recortado['ganancia_final'] <= 1500000),
        (df_recortado['ganancia_final'] > 1500000)
    ]
    opciones_tramos = ['Menos de $350.000', 'Entre $350.000 y $700.000', 'Entre $700.000 y $1.500.000', 'Más de $1.500.000']
    df_recortado['tramos_ganancias'] = np.select(condiciones, opciones_tramos, default='Sin Información')

# Mapeo textual directo para las preguntas estructurales del Dashboard
mapa_ramas = {
    1: 'Agricultura y Pesca', 2: 'Minería', 3: 'Manufactura', 4: 'Electricidad y Gas',
    5: 'Agua y Desechos', 6: 'Construcción', 7: 'Comercio Minorista/Mayorista', 8: 'Transporte',
    9: 'Alojamiento y Comida', 10: 'Comunicaciones', 11: 'Finanzas', 12: 'Actividades Inmobiliarias',
    13: 'Profesional y Técnica', 14: 'Servicios Administrativos', 15: 'Enseñanza',
    16: 'Salud y Asistencia Social', 17: 'Artes y Entretenimiento', 18: 'Otras actividades de servicios',
    19: 'Hogares como Empleadores'
}
mapa_motivacion = {
    1: 'Oportunidad de negocio', 2: 'Complementar ingreso familiar', 3: 'Necesidad (Falta de empleo)',
    4: 'Mayor independencia', 5: 'Razones de salud/edad', 6: 'Compatibilizar hogar/cuidados',
    7: 'Tradición familiar', 8: 'Acceso a instalaciones/maquinaria', 9: 'Otra razón'
}

if 'rama_economica' in df_recortado.columns: df_recortado['rama_economica'] = df_recortado['rama_economica'].map(mapa_ramas).fillna('Otros sectores')
if 'motivacion' in df_recortado.columns: df_recortado['motivacion'] = df_recortado['motivacion'].map(mapa_motivacion).fillna('Otras razones')

# Rellenamos de forma genérica variables complementarias en caso de códigos vacíos para asegurar consistencia
if 'tramo_etario' in df_recortado.columns:
    df_recortado['tramo_etario'] = df_recortado['tramo_etario'].map({1: '15 a 29 años', 2: '30 a 44 años', 3: '45 a 59 años', 4: '60 años o más'}).fillna('Adulto Medio')
if 'nivel_educativo' in df_recortado.columns:
    df_recortado['nivel_educativo'] = df_recortado['nivel_educativo'].map({1: 'Educación Básica', 2: 'Educación Media', 3: 'Técnica Superior', 4: 'Universitaria'}).fillna('Educación Media')
if 'lugar_trabajo' in df_recortado.columns:
    df_recortado['lugar_trabajo'] = df_recortado['lugar_trabajo'].map({1: 'En el propio hogar', 2: 'Local u oficina comercial', 3: 'En la vía pública', 4: 'A domicilio'}).fillna('Instalación Independiente')
if 'financiamiento_inicial' in df_recortado.columns:
    df_recortado['financiamiento_inicial'] = df_recortado['financiamiento_inicial'].map({1: 'Ahorros propios', 2: 'Préstamo bancario', 3: 'Fondos estatales (Sercotec/Corfo)', 4: 'Préstamo familiar'}).fillna('Recursos Propios')

# Limpieza final de estructura básica
df_recortado.drop(columns=['informalidad_cod'], inplace=True, errors='ignore')
df_recortado.dropna(subset=['region', 'sexo', 'informalidad'], inplace=True)

# 5. Guardar y forzar descarga
df_recortado.to_csv("dataset_limpio.csv", index=False)
print(f"Dataset guardado con éxito")
files.download("dataset_limpio.csv")

Dataset original cargado con 5503 filas y 669 columnas.

Dataset guardado con éxito


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("dataset_limpio.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_recortado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3844 entries, 0 to 7167
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   region          3844 non-null   object
 1   sexo            3844 non-null   object
 2   informalidad    3844 non-null   object
 3   rama_economica  3844 non-null   object
 4   motivacion      3838 non-null   object
dtypes: object(5)
memory usage: 180.2+ KB
